In [68]:
import json, numpy as np, pandas as pd
from pathlib import Path
from unittest import mock

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import pcmci_sweep as ps
import causal_effect as ce

RUN_DIR = Path("pcmci_runs/runs_agg_v2")
EFF_DIR = Path("effects_runs/assert")
XVARS   = ["ratio_1", "ratio_2", "ratio_3"]        # ratio_4 has no identifiable effect
WINDOW  = "3h"

# every pos/ratio run that has saved effect estimates, coarse -> fine
RUNS = sorted((p for p in RUN_DIR.glob("*_pos_ratio_*.npz")
               if list(EFF_DIR.glob(p.stem + "__ratio_*__TMS_nox_clean.npz"))),
              key=lambda p: pd.Timedelta(json.loads(str(np.load(p)["config"]))["FREQ"]))
print("runs:", [p.name for p in RUNS])

# every run here is the same recipe at a different resolution, so the frame recipe
# is restored once and the parquet is read once -- resampling is the only per-run work
_, vn0, cfg0 = ce.load_run(RUNS[0])
assert (cfg0["START"], cfg0["END"]) == (ps.START, ps.END), "sweep START/END drifted"
ce.restore_recipe(cfg0, vn0)

if "d" not in globals():                            # slow; survives a re-run of this cell
    d, flows = ps.load_raw()

TARGET = ps.TARGET_CLEAN


runs: ['13_pos_ratio_3_2h_c15ff40f.npz', '5_pos_ratio_5_3.5h_1e40bb22.npz', '2_pos_ratio_10_3.5h_86ebf4ad.npz', '7_pos_ratio_15_3.5h_9192f0f2.npz', '15_pos_ratio_30_3.5h_92c0cd78.npz']
composites restored from the cached run: arch_3_4=['ARCH #3', 'ARCH #4'], bt_temp=['TEMP #08', 'TEMP #09', 'TEMP #10', 'MELTER BT #11']


In [69]:
BUMP = np.log1p(ce.STEP_PCT / 100)      # psi is stored per +1% of X

def kernel(path):
    """psi -> the multiplier on a raw differenced-log move of X."""
    e   = ce.load_effect(path)
    psi = np.nan_to_num(e["psi"], nan=0.0)          # not-identifiable lags contribute 0
    return e["meta"]["x"], psi / (BUMP if e["meta"]["islog"] else 1.0), e

def conv(dx, k):
    """dNOx_hat[t] = sum_tau k[tau]*dx[t-tau]; NaN wherever the lag window hit one."""
    v   = dx.to_numpy(float)
    bad = ~np.isfinite(v)
    out = np.convolve(np.where(bad, 0.0, v), k)[:len(v)]
    hit = np.convolve(bad.astype(float), (k != 0).astype(float))[:len(v)] > 0
    out[hit] = np.nan
    return pd.Series(out, index=dx.index)

RES = {}
for run in RUNS:
    graph, vn, cfg = ce.load_run(run)
    if vn != vn0:
        raise SystemExit(f"{run.name} has different variables -- restore_recipe would be wrong")

    R = ps.resample(d, flows, cfg["FREQ"])
    # build_frame ends in .diff(); we want levels too, so neutralise that one call
    # instead of reimplementing the column recipe and letting it drift from the sweep
    with mock.patch.object(pd.DataFrame, "diff", lambda self, *a, **k: self):
        L = ps.build_frame(R, cfg["flows"], cfg["ratio"])
    C = L.diff()
    assert list(C.columns) == vn

    contrib, TOTAL = {}, {}
    for p in sorted(EFF_DIR.glob(run.stem + "__ratio_*__TMS_nox_clean.npz")):
        x, k, e = kernel(p)
        if x not in XVARS or (k != 0).sum() == 0:
            continue                                 # empty template -> nothing to predict
        contrib[x] = conv(C[x], k)                   # this ratio's own prediction
        TOTAL[x]   = float(np.nansum(e["psi"]))      # total level effect per +1%

    RES[cfg["FREQ"]] = dict(cfg=cfg, L=L, C=C, vn=vn, contrib=contrib, TOTAL=TOTAL)
    print(f"{cfg['FREQ']:>6}  {len(L)} bins  "
          + "  ".join(f"{x} {TOTAL[x]:+.2f}" for x in sorted(TOTAL)) + "  NOx per +1%")

FREQS = list(RES)


  3min  45120 bins  ratio_1 -0.59  NOx per +1%
  5min  27072 bins  ratio_1 +4.10  ratio_3 +4.49  NOx per +1%
 10min  13536 bins  ratio_1 +4.19  ratio_2 +4.34  ratio_3 +2.49  NOx per +1%
 15min  9024 bins  ratio_2 +4.97  ratio_3 +2.45  NOx per +1%
 30min  4512 bins  ratio_2 +3.52  ratio_3 +4.69  NOx per +1%


In [70]:
# --- all in human units -----------------------------------------------------
MOVE_PCT  = 1.5        # a ratio move smaller than this is not worth showing
MOVE_SPAN = "30min"    # ...and it has to happen within this long
BEFORE    = "30min"    # NOx baseline over this stretch before the move
AFTER     = "30min"    # NOx response over this stretch after it (psi is done by ~30min)
MIN_GAP   = "2h"       # two moves closer than this are one event, not two
QUIET     = 4.0        # rivals may move at most this many sigma
MATCH     = (0.5, 2.0) # observed/expected band that counts as "NOx followed"
# ----------------------------------------------------------------------------

def others_for(C, vn, x):
    """everything that could steal credit from x -- the OTHER ratios included.
    oil_* is excluded: ratio = oxy - oil in logs, so it moves by construction."""
    cols = [c for c in vn if c not in (x, TARGET)
            and not c.startswith(("WEATHER_", "oil_"))]
    return (C[cols] / C[cols].std()).abs().max(axis=1)

rows = []
for freq, B in RES.items():
    L, C, step = B["L"], B["C"], pd.Timedelta(freq)
    # spans are given in time but applied in bins, so 3min and 30min runs both
    # get a usable baseline instead of the coarse ones silently finding nothing
    n_move = max(1, round(pd.Timedelta(MOVE_SPAN) / step))
    n_bef  = max(2, round(pd.Timedelta(BEFORE)    / step))
    n_aft  = max(2, round(pd.Timedelta(AFTER)     / step))
    n_gap  = max(1, round(pd.Timedelta(MIN_GAP)   / step))
    noxv   = L[TARGET].to_numpy(float)

    for x, total in B["TOTAL"].items():
        lv   = L[x].to_numpy(float)
        zo   = others_for(C, B["vn"], x).to_numpy(float)
        pct  = np.full(len(lv), np.nan)
        pct[n_move:] = np.expm1(lv[n_move:] - lv[:-n_move]) * 100

        order, taken = np.argsort(-np.abs(np.nan_to_num(pct))), []
        for i in order:
            p = pct[i]
            if not np.isfinite(p) or abs(p) < MOVE_PCT:
                break                                    # sorted: nothing left is big enough
            if any(abs(i - u) < n_gap for u in taken):
                continue                                 # same ramp, already have it
            i0 = i - n_move                              # move starts here
            if i0 - n_bef < 0 or i + 1 + n_aft > len(lv):
                continue
            base, resp = noxv[i0 - n_bef: i0 + 1], noxv[i + 1: i + 1 + n_aft]
            if np.isnan(base).any() or np.isnan(resp).any():
                continue                                 # dropout / burner cleaning
            taken.append(i)
            rows.append(dict(
                res      = freq,
                var      = x,
                time     = L.index[i],                   # move ends here
                move_pct = p,
                expected = total * p,
                observed = float(resp.mean() - base.mean()),
                others   = float(np.nanmax(zo[i0: i + 1 + n_aft])),
            ))

EV = pd.DataFrame(rows)
EV["match"] = EV.observed / EV.expected               # 1.0 = exactly as estimated
GOOD = EV[EV.match.between(*MATCH) & (EV.others <= QUIET)].copy()
GOOD["size"] = GOOD.expected.abs()

print(EV.groupby(["res", "var"]).size().unstack(fill_value=0), "\n   moves found\n")
print(GOOD.groupby(["res", "var"]).size().unstack(fill_value=0), "\n   NOx followed")


var    ratio_1  ratio_2  ratio_3
res                             
10min       67       73       70
15min        0       68       65
30min        0       35       33
3min        72        0        0
5min        70        0       70 
   moves found

var    ratio_1  ratio_2  ratio_3
res                             
10min       13       17        9
15min        0       16        7
30min        0        9        4
5min        11        0        8 
   NOx followed


In [71]:
PER_CELL = 3          # candidates to look at for each res x ratio combination
PRE      = pd.Timedelta("30min")     # lead-in shown before the move starts

G = GOOD.copy()
G["quality"] = G["size"] / (1 + (G.match - 1).abs())     # big, and close to 1:1
pick = (G.sort_values("quality", ascending=False)
          .groupby(["res", "var"], group_keys=False).head(PER_CELL))

BEST = pick.assign(
    # 45min + 30min is not a whole number of bins at 10/15/30min, so snap the
    # window to the run's own grid instead of landing between two samples
    start = [(t - pd.Timedelta(MOVE_SPAN) - PRE).floor(f)
             for t, f in zip(pick.time, pick.res)],
).assign(
    end = lambda f: f.start + pd.Timedelta(WINDOW),
).sort_values(
    ["var", "res", "quality"], ascending=[True, True, False],
    key=lambda s: s.map(lambda v: pd.Timedelta(v)) if s.name == "res" else s,
).reset_index(drop=True)

BEST[["var", "res", "start", "move_pct", "expected", "observed", "match", "others"]].round(2)


,var,res,start,move_pct,expected,observed,match,others
0,ratio_1,5min,2025-08-28 16:20:00,-4.54,-18.62,-24.93,1.34,2.54
1,ratio_1,5min,2025-08-01 03:55:00,4.30,17.66,22.62,1.28,2.20
2,ratio_1,5min,2025-10-01 20:55:00,-3.90,-16.01,-23.72,1.48,1.62
3,ratio_1,10min,2025-09-20 02:40:00,-4.22,-17.71,-19.59,1.11,2.18
4,ratio_1,10min,2025-08-01 03:50:00,4.21,17.64,22.85,1.29,1.29
5,ratio_1,10min,2025-10-01 20:50:00,-3.77,-15.83,-23.68,1.50,1.82
6,ratio_2,10min,2025-07-31 23:20:00,4.18,18.14,17.73,0.98,1.21
7,ratio_2,10min,2025-09-16 10:50:00,4.63,20.10,24.71,1.23,1.47
8,ratio_2,10min,2025-08-05 00:50:00,4.31,18.74,22.53,1.20,3.45
9,ratio_2,15min,2025-09-16 10:45:00,4.74,23.59,24.14,1.02,1.28


In [72]:
RATIO_COLOR = {"ratio_1": "royalblue", "ratio_2": "orange", "ratio_3": "purple"}

def nan_runs(s):
    """(start, end) of every NaN stretch -- NOx dropouts and burner cleaning."""
    bad, out, i = s.isna().to_numpy(), [], 0
    while i < len(bad):
        if bad[i]:
            j = i
            while j + 1 < len(bad) and bad[j + 1]:
                j += 1
            out.append((s.index[i], s.index[j])); i = j + 1
        else:
            i += 1
    return out


def show(i, pad="1h"):
    """One candidate from BEST, by row number."""
    r = BEST.loc[i]
    B = RES[r.res]
    L, focus = B["L"], r["var"]
    start, end = r.start, r.end
    lo, hi = start - pd.Timedelta(pad), end + pd.Timedelta(pad)
    sl = slice(lo, hi)

    obs = L[TARGET][sl]
    # this ratio alone, not the sum -- the slide shows one input, so the dashed
    # line must not be partly explained by a ratio that is off-screen
    anchor = obs.asof(start)
    if pd.isna(anchor):
        anchor = obs.dropna().iloc[0]
    pred = anchor + B["contrib"][focus][start:hi].cumsum()


    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=obs.index, y=obs, mode='lines', name='NOx 실측',
                             line=dict(color='black', width=2)), secondary_y=False)
    fig.add_trace(go.Scatter(x=pred.index, y=pred, mode='lines',
                             name=f'NOx 예측 ({focus} 인과효과만 반영)',
                             line=dict(color='crimson', width=1.8)),
                  secondary_y=False)
    xin = [c for c in XVARS if c in L.columns]
    for x in sorted(xin, key=lambda c: c == focus):      # focus drawn last = on top
        lead = x == focus
        fig.add_trace(go.Scatter(
            x=L.index[L.index.slice_indexer(lo, hi)], y=np.exp(L[x][sl]),
            mode='lines', name=x + ("　(대상)" if lead else ""),
            line=dict(color=RATIO_COLOR.get(x, 'green'),
                      width=2.2 if lead else 1.0,
                      dash='solid' if lead else 'dot'),
            opacity=1.0 if lead else 0.45,
        ), secondary_y=True)


    for s, e in nan_runs(obs):
        fig.add_vrect(x0=s, x1=e, fillcolor="gray", opacity=0.2, layer="below")
    fig.add_vrect(x0=start, x1=end, fillcolor="tomato", opacity=0.08, layer="below")
    fig.add_vline(x=r.time - pd.Timedelta(MOVE_SPAN), line=dict(color='green', width=2, dash='dash'))
    fig.add_vline(x=r.time, line=dict(color='green', width=2, dash='dash'))

    fig.update_layout(
        title=(f"[{i}] {focus} @ {r.res}　{start:%Y-%m-%d %H:%M}　　"
               f"{r.move_pct:+.1f}%　예상 {r.expected:+.1f}　실측 {r.observed:+.1f}"
               f"　(match {r.match:.2f}, others {r.others:.1f}σ)"),
        xaxis_title="시간", template="plotly_white",
        yaxis=dict(title="NOx", color='black'),
        yaxis2=dict(title="산소/연료 비율", overlaying="y", side="right",
                    color=RATIO_COLOR.get(focus, 'green'), showgrid=False),
        legend=dict(x=1.02, y=1, itemclick="toggle", itemdoubleclick="toggleothers"),
        hovermode="x unified",
    )
    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True, secondary_y=False)
    fig.update_yaxes(showgrid=False, secondary_y=True)
    return fig


In [75]:
BROWSE = "ratio_1"        # set to a ratio, a resolution ("10min"), or None for all

sel = BEST.index if BROWSE is None else BEST.index[
    (BEST["var"] == BROWSE) | (BEST["res"] == BROWSE)]
print(f"{len(sel)} candidates: {list(sel)}")

for i in sel:
    show(i).show()


6 candidates: [0, 1, 2, 3, 4, 5]
